In [ ]:
!pip install -q evaluate transformers bitsandbytes accelerate gradio rouge_score

  Preparing metadata (setup.py) ... done


In [ ]:
import json, os, random
import evaluate
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE       = "/content/drive/MyDrive/arxiv-llm-project"
EXTRACTED_FILE   = f"{DRIVE_BASE}/data/processed/extracted_papers.jsonl"
EVAL_HOLDOUT     = f"{DRIVE_BASE}/eval/holdout_50.jsonl"
RESULTS_BASE     = f"{DRIVE_BASE}/eval/results_base.json"

BASE_MODEL_ID    = "meta-llama/Llama-3.2-3B-Instruct"
HOLDOUT_SIZE     = 50
SEED             = 42

os.makedirs(f"{DRIVE_BASE}/eval", exist_ok=True)

Mounted at /content/drive


In [ ]:
random.seed(SEED)

all_papers = []
with open(EXTRACTED_FILE, "r") as f:
    for line in f:
        all_papers.append(json.loads(line.strip()))

holdout = random.sample(all_papers, HOLDOUT_SIZE)

with open(EVAL_HOLDOUT, "w") as f:
    for p in holdout:
        f.write(json.dumps(p) + "\n")

print(f"Saved {len(holdout)} holdout papers → {EVAL_HOLDOUT}")
print(f"Remaining for training: {len(all_papers) - len(holdout)}")

Saved 50 holdout papers → /content/drive/MyDrive/arxiv-llm-project/eval/holdout_50.jsonl
Remaining for training: 4950


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"\nLoading base model: {BASE_MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print("Model loaded ✓")


Loading base model: meta-llama/Llama-3.2-3B-Instruct


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded ✓


In [ ]:
SUMMARIZE_PROMPT = """You are a research assistant. Given the title and body of an ML paper,
write a concise abstract-style summary (3-4 sentences) covering: main problem, proposed method,
key results.

Title: {title}

Paper excerpt:
{body}

Summary:"""

def generate_summary(paper: dict, max_new_tokens: int = 200) -> str:
    body_excerpt = (paper.get("body_text") or paper.get("abstract") or "")[:1500]
    prompt = SUMMARIZE_PROMPT.format(title=paper["title"], body=body_excerpt)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,      # greedy for reproducibility
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [ ]:
rouge = evaluate.load("rouge")

predictions = []
references  = []
per_paper   = []

print(f"\nGenerating summaries for {HOLDOUT_SIZE} holdout papers...")
for paper in tqdm(holdout):
    pred = generate_summary(paper)
    ref  = paper.get("abstract", "")   # ground-truth reference

    predictions.append(pred)
    references.append(ref)
    per_paper.append({
        "paper_id":   paper["paper_id"],
        "title":      paper["title"],
        "reference":  ref,
        "prediction": pred,
    })

results = rouge.compute(predictions=predictions, references=references, use_stemmer=True)
print("\n── ROUGE Scores (Base Model, No RAG) ───────────────────")
for k, v in results.items():
    print(f"  {k}: {v:.4f}")


Generating summaries for 50 holdout papers...



  0%|          | 0/50 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)

100%|██████████| 50/50 [11:42<00:00, 14.05s/it]



── ROUGE Scores (Base Model, No RAG) ───────────────────
  rouge1: 0.5020
  rouge2: 0.2641
  rougeL: 0.3154
  rougeLsum: 0.3376


In [ ]:
output = {
    "model":        BASE_MODEL_ID,
    "mode":         "base_no_rag",
    "holdout_size": HOLDOUT_SIZE,
    "scores":       results,
    "per_paper":    per_paper,
}

with open(RESULTS_BASE, "w") as f:
    json.dump(output, f, indent=2)

print(f"\nResults saved → {RESULTS_BASE}")
print(f"ROUGE-L baseline: {results['rougeL']:.4f}  (target to beat: > 0.30 after RAG)")


Results saved → /content/drive/MyDrive/arxiv-llm-project/eval/results_base.json
ROUGE-L baseline: 0.3154  (target to beat: > 0.30 after RAG)


In [ ]:
import gradio as gr

def gradio_summarize(paper_title: str, paper_text: str) -> str:
    if not paper_title.strip() or not paper_text.strip():
        return "Please provide both a title and paper text."
    paper = {"title": paper_title, "body_text": paper_text, "abstract": ""}
    return generate_summary(paper)

with gr.Blocks(title="Research Assistant — Base Model") as demo:
    gr.Markdown("## Research Assistant\n_Base model (no fine-tuning, no RAG) - Milestone 2.5 stub_")

    with gr.Row():
        with gr.Column():
            title_input = gr.Textbox(label="Paper Title", placeholder="e.g. Attention Is All You Need")
            text_input  = gr.Textbox(label="Paper Excerpt (paste body text)", lines=10,
                                     placeholder="Paste the first few paragraphs here...")
            btn         = gr.Button("Summarize", variant="primary")
        with gr.Column():
            output_box  = gr.Textbox(label="Generated Summary", lines=8)

    btn.click(fn=gradio_summarize, inputs=[title_input, text_input], outputs=output_box)

    gr.Markdown("""
    **Modes coming in Phase 3:**
    - 📝 Summarize (this tab)
    - ❓ Q&A with RAG retrieval
    - 🔗 Related work finder
    """)

print("\n✅ Launching Gradio stub wired to base model...")
demo.launch(share=True)


✅ Launching Gradio stub wired to base model...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://09eedf0b129082d735.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
